# Import's & Preprocessing

In [ ]:
import preprocess as pp
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

## Delete Warning Messages
import warnings
warnings.filterwarnings("ignore")

data = pp.DataPreprocessor()
merged_df = data.get_merged_data()

## Display Load Data

In [ ]:
display(merged_df)

# David Arzumanyan

In [ ]:

merged_df = pd.merge(
    marketData,
    Crime2024,
    on="CITY",       # common column name
    how="left"      # join type: inner keeps only cities that exist in both datasets
)


In [ ]:
merged_df[merged_df["CITY"] == "Watertown"]

In [ ]:
marketData = marketData[marketData["STATE"] == "Massachusetts"] # Filtering MA data only
marketData["PERIOD_BEGIN"] = pd.to_datetime(marketData["PERIOD_BEGIN"])
marketData["PERIOD_END"] = pd.to_datetime(marketData["PERIOD_END"])
marketData = marketData[marketData["PERIOD_BEGIN"].dt.year > 2014] # Filtering data from 2015

In [ ]:
marketData[(marketData["CITY"] == "Watertown") & (marketData["HOMES_SOLD"] > 50)]

# Joe Tivnan

In [ ]:

# take the merketData and remove the 'IS_SEASONALLY_ADJUSTED' column
# slim_marketData = marketData.drop(columns=['IS_SEASONALLY_ADJUSTED',
#                                           'OFF_MARKET_IN_TWO_WEEKS',
#                                           'OFF_MARKET_IN_TWO_WEEKS_MOM',
#                                           'OFF_MARKET_IN_TWO_WEEKS_YOY',
#                                           'MEDIAN_SALE_PRICE',
#                                           'MEDIAN_SALE_PRICE_MOM',
#                                           'MEDIAN_SALE_PRICE_YOY',
#                                           'LAST_UPDATED'])


# download the slim_marketData
# slim_marketData.to_csv("Data/slim_massachusetts_market_data.csv", index=False)

In [ ]:
display(marketData['AVG_SALE_TO_LIST'])

# plot a scatter plot in seaborn of AVG_SALE_TO_LIST vs DATE, using YYYY-MM-DD format for the BEGIN_PERIOD column
import seaborn as sns
import matplotlib.pyplot as plt

# Convert the 'BEGIN_PERIOD' column to datetime format
marketData['PERIOD_BEGIN'] = pd.to_datetime(marketData['PERIOD_BEGIN'])

# Create the scatter plot
plt.figure(figsize=(12, 6))
sns.scatterplot(data=marketData, x='PERIOD_BEGIN', y='AVG_SALE_TO_LIST')
plt.title('Average Sale to List Ratio Over Time')
plt.xlabel('Date')
plt.ylabel('Average Sale to List Ratio')
plt.xticks(rotation=45)
plt.tight_layout()

# add a mean line to the scatter plot and add a legend
mean_value = marketData['AVG_SALE_TO_LIST'].mean()
plt.axhline(mean_value, color='red', linestyle='--', label='Mean')
plt.legend()
plt.show()

# plot just the values from 2020
marketData_2020 = marketData[marketData['PERIOD_BEGIN'].dt.year == 2020]
plt.figure(figsize=(12, 6))
sns.scatterplot(data=marketData_2020, x='PERIOD_BEGIN', y='AVG_SALE_TO_LIST')
plt.title('Average Sale to List Ratio in 2020')
plt.xlabel('Date')
plt.ylabel('Average Sale to List Ratio')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# create a violin plot of all the distributions for 2020 by month
marketData_2020['MONTH'] = marketData_2020['PERIOD_BEGIN'].dt.month
plt.figure(figsize=(12, 6))
sns.violinplot(data=marketData_2020, x='MONTH', y='AVG_SALE_TO_LIST')
plt.title('Distribution of Average Sale to List Ratio by Month in 2020')
plt.xlabel('Month')
plt.ylabel('Average Sale to List Ratio')
plt.tight_layout()
plt.show()

# Patrick Keufack

In [ ]:
## Phase one Data cleaning is done above, so jump to Phase 2

## Checking our dataset structure and missing data before proceeding to correlation analysis(Phase 2)

## ---- Phase 2 (fixed): Feature Engineering & Exploration ----
import matplotlib.pyplot as plt
import numpy as np

#  new features using real column names
merged_df['log_price'] = np.log1p(merged_df['MEDIAN_LIST_PRICE'])

# Just to make things clearer, renaming a few for easy use later
merged_df.rename(columns={'MEDIAN_LIST_PRICE': 'price', 'MEDIAN_PPSF': 'price_per_sqft'}, inplace=True)

# Basic visualizations
plt.hist(merged_df['price'], bins=30, color='skyblue')
plt.title("Median Housing Price Distribution")
plt.xlabel("Price ($)")
plt.ylabel("Frequency")
plt.show()

plt.hist(merged_df['price_per_sqft'], bins=30, color='orange')
plt.title("Price per Square Foot Distribution")
plt.xlabel("Price per sqft ($)")
plt.ylabel("Count")
plt.show()

In [ ]:
## ---- Phase 2 continued: Correlation Analysis ----
import seaborn as sns
import matplotlib.pyplot as plt

# Select only numeric columns
num_df = merged_df.select_dtypes(include=[np.number])

# Compute correlation with price
corr = num_df.corr(numeric_only=True)
price_corr = corr['price'].sort_values(ascending=False)

# Show top correlated features with price
print(price_corr.head(10))

# Heatmap for visual correlation
plt.figure(figsize=(10,6))
sns.heatmap(corr, cmap="coolwarm", annot=False)
plt.title("Correlation Heatmap of Numeric Features")
plt.show()

# Victor

In [ ]:
housing_cities = set(marketData['CITY'].unique())
crime_cities = set(crime_all['CITY'].unique())

# Matches
matching_cities = housing_cities & crime_cities

# Mismatches
housing_not_in_crime = housing_cities - crime_cities
crime_not_in_housing = crime_cities - housing_cities

print("Matching Cities:", len(matching_cities))
print("Housing but not Crime:", len(housing_not_in_crime))
print("Crime but not Housing:", len(crime_not_in_housing))


In [ ]:
price_trends = merged_df.groupby('year')[['MEDIAN_LIST_PRICE','AVG_SALE_TO_LIST']].mean().reset_index()

import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

fig, ax1 = plt.subplots(figsize=(10,6))

# Left axis → List Price
ax1.plot(price_trends['year'], price_trends['MEDIAN_LIST_PRICE'], color='tab:blue', label='Median List Price')
ax1.set_ylabel("Median List Price (USD)", color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# Right axis → Sale-to-List Ratio
ax2 = ax1.twinx()
ax2.plot(price_trends['year'], price_trends['AVG_SALE_TO_LIST'], color='tab:orange', label='Sale-to-List Ratio')
ax2.set_ylabel("Sale-to-List Ratio", color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')

plt.title("Massachusetts Housing Market Trends Over Time")
plt.xlabel("Year")
fig.tight_layout()
plt.show()


